# CS781 Phase 4 

In [1]:
import os, sys, json, time, glob, subprocess, shutil
import numpy as np, pandas as pd
WORK = "/kaggle/working"; sys.path.insert(0, WORK)
!pip install -q ijson
def find(name, root="/kaggle/input"):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root, followlinks=True) if name in fs]
    return sorted(hits, key=os.path.getmtime)
import torch
def is_hybrid(p):
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        sd = ck.get("state_dict", ck.get("model_state", ck)) if isinstance(ck, dict) else {}
        return any(".mixer.dw_weight" in k for k in sd)
    except Exception: return False
cands = [p for p in find("model.pt") + find("checkpoint.pt") if is_hybrid(p)]
INIT = (cands or [None])[-1]
TOK  = None
if INIT:
    near = os.path.join(os.path.dirname(INIT), "bpe_tokenizer.json")
    TOK = near if os.path.exists(near) else None
TOK = TOK or (find("bpe_tokenizer.json") or [None])[-1]
assert INIT and TOK, f"attach your Phase-1 output dataset. /kaggle/input has: {os.listdir('/kaggle/input')}"
print("INIT:", INIT, "\nTOK :", TOK)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 3.3 MB/s eta 0:00:00
INIT: /kaggle/input/datasets/luckyj12/hybrid-part-1/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/checkpoint.pt 
TOK : /kaggle/input/datasets/luckyj12/hybrid-part-1/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/bpe_tokenizer.json
GPUs: 2 ['Tesla T4', 'Tesla T4']


In [2]:
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HuggingFace")
DATA = f"{WORK}/p4data"; os.makedirs(DATA, exist_ok=True)
repo_files = [f for f in list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=hf_token) if "Phase_4" in f]
print("\n".join(repo_files))
for w in ["train.json", "dev.json", "test.json"]:
    hit = [f for f in repo_files if f.endswith("/" + w)]
    assert len(hit) == 1, f"{w}: expected exactly one match in the listing above, got {hit}"
    if not os.path.exists(f"{DATA}/{w}"):
        shutil.copy(hf_hub_download("Exploration-Lab/CS-781-Capstone", hit[0], repo_type="dataset", token=hf_token), f"{DATA}/{w}")
print({f: round(os.path.getsize(f"{DATA}/{f}") / 2**20) for f in os.listdir(DATA)}, "MB")

for w in ["train.json", "dev.json", "test.json"]:
    print(w, "->", open(f"{DATA}/{w}", encoding="utf-8").read(600).replace("\n", " ")[:600], "\n")

Phase_4_CJPE/Test/test.json
Phase_4_CJPE/Train/dev.json
Phase_4_CJPE/Train/train.json


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

{'test.json': 34, 'train.json': 703, 'dev.json': 43} MB
train.json -> [   {     "id": "2020_1",     "text": " Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters and thereby affirming the decision of the Single Judge of the High Court passed on 12.03.2014 in Writ Petition No.20650  W  of 2013 which in turn had found Sections 8, 10, 11 and 12 of the West Bengal Madrasah Service Commission Act, 2008  the Commission Act, for short  to be ultra vires. 1 The High Court of Judicature at Calcutta Civil Appeal No.5808 of 201 

dev.json -> [{"id": "1989_75", "text": "civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.10.1987 of t\nhe\ncentral administrative tribunal chandigarh in appln. t.\n--\n1055 of 1986.\nwith\n slp  civil  number 6998 of 1988 \n\nc. mahajan mrs. indu goswami c.v. subba rao\np.\

In [3]:
%%writefile cjpe_hybrid.py

"""
cjpe_hybrid.py --  CS781 Phase 4 (Court Judgment Prediction, binary 0/1, metric = macro-F1) on top of YOUR Phase-1 HYBRID model (gated causal conv mixers + sparse RoPE attention, causal).

One run = one adaptation method.  Examples:
  python cjpe_hybrid.py --method full   --data_dir DATA --tokenizer bpe_tokenizer.json --init model.pt --out runs/full
  python cjpe_hybrid.py --method lorma  ... --out runs/lorma
  python cjpe_hybrid.py --method kd     ... --teacher runs/full/best.pt --out runs/kd

Document representation (--view):
  tail      last `window` tokens (default 1024)                      <- cheapest; the court's reasoning sits at the end
  head      first `window` tokens
  headtail  first `head_tok` + last `window-head_tok` tokens
  chunks    up to `max_chunks` windows of `window` tokens (stride `stride`), taken from the END (--pick tail) or spread over
            the whole document (--pick spread); every chunk goes through the Phase-1 trunk, chunk vectors are aggregated
            by --agg attn|mean|last (+ a learned "chunks-from-the-end" embedding) and fed to a linear head.
Only the first `head_words` and last `tail_words` words of each case are tokenised (saves hours; both are cached).

Sections
  A. Phase-1 tokenizer + hybrid model code (copied from your hybrid Phase-1 notebook; [P4-EDIT] = optional layer_types)
  B. Data (streaming JSON reader, tokenise + cache, views/chunks, collate, metrics)
  C. Model wrapper + adaptation methods: full / partial / bitfit / lora / lorma / adalora / adapter / ia3 / kd
  D. Training, threshold calibration, inference, prediction writer
"""
import argparse, hashlib, json, math, os, pickle, random, sys, time, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PAD_ID = 256          
BOS_ID = 258          
EOS_ID = 259          
CHUNK_CAP = 16        




class ByteLevelBPETokenizer:
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = 256 + i
            self.special_to_id[tok] = tid
            self.id_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self):
        return len(self.id_to_bytes)

    @staticmethod
    def _pair_counts(sequences):
        counts = {}
        for seq, freq in sequences:
            for pair in zip(seq, seq[1:]):
                counts[pair] = counts.get(pair, 0) + freq
        return counts

    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                out.append(new_id); i += 2
            else:
                out.append(seq[i]); i += 1
        return out

    def train(self, text, vocab_size, max_words=100000, verbose=False):   
        num_merges = vocab_size - self.vocab_size
        word_freq = {}
        for word in text.split():
            word_freq[word] = word_freq.get(word, 0) + 1
        sequences = [[list(w.encode("utf-8")), f] for w, f in word_freq.items()]
        next_id = self.vocab_size
        for _ in range(num_merges):
            counts = self._pair_counts(sequences)
            if not counts: break
            best = max(counts, key=counts.get)
            self.merges[best] = next_id
            self.id_to_bytes[next_id] = self.id_to_bytes[best[0]] + self.id_to_bytes[best[1]]
            for e in sequences: e[0] = self._apply_merge(e[0], best, next_id)
            next_id += 1
        return self

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": {f"{a},{b}": n for (a, b), n in self.merges.items()}}, f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok


class RMSNorm(nn.Module):
    """RMS normalization without mean subtraction."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)          
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos[:, :, offset:offset + T, :].to(x.dtype)
        sin = self.sin[:, :, offset:offset + T, :].to(x.dtype)
        return x * cos + rotate_half(x) * sin


class GeGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.gelu(self.gate(x)) * self.up(x)))


class GatedCausalConvMixer(nn.Module):
    """
    Local token mixer (Conformer / ConvBERT / gated-conv style) made causal for LM:
        x -> [value | gate] projection -> causal depthwise Conv1d(k) on value -> value * silu(gate) -> out proj
    Receptive field per layer = kernel size; no positional embedding needed (convolution is position-aware).
    Cost is O(T * k) instead of O(T^2).
    """
    def __init__(self, d_model, kernel_size, dropout):
        super().__init__()
        self.k = kernel_size
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.dw_weight = nn.Parameter(torch.empty(d_model, 1, kernel_size))
        nn.init.normal_(self.dw_weight, std=0.02)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, cache=None, offset=0):
        B, T, C = x.shape
        a, g = self.in_proj(x).chunk(2, dim=-1)
        a = a.transpose(1, 2)                                     
        if cache is None:
            state = a.new_zeros(B, C, self.k - 1)                 
        else:
            state = cache["state"]
        a_pad = torch.cat([state, a], dim=2)
        y = F.conv1d(a_pad, self.dw_weight.to(a.dtype), groups=C) 
        new_cache = {"state": a_pad[:, :, a_pad.size(2) - (self.k - 1):]}
        y = y.transpose(1, 2) * F.silu(g)
        return self.dropout(self.out_proj(y)), new_cache


class CausalSelfAttention(nn.Module):
    """Global causal multi-head attention with RoPE; used sparsely (every ATTN_EVERY-th layer)."""
    def __init__(self, d_model, n_heads, rope, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope, self.dropout = rope, dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))
        q, k = self.rope(q, offset), self.rope(k, offset)
        if cache is None:                                         
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:                                                     
            assert T == 1
            k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out)), {"k": k, "v": v}


class HybridBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope, dropout, use_attention, conv_kernel):
        super().__init__()
        self.use_attention = use_attention
        self.norm_mix = RMSNorm(d_model)
        self.mixer = CausalSelfAttention(d_model, n_heads, rope, dropout) if use_attention \
            else GatedCausalConvMixer(d_model, conv_kernel, dropout)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = GeGLU(d_model, d_ff, dropout)

    def forward(self, x, cache=None, offset=0):
        h, new_cache = self.mixer(self.norm_mix(x), cache=cache, offset=offset)
        x = x + h
        x = x + self.ffn(self.norm_ffn(x))
        return x, new_cache


class HybridConvAttnLM(nn.Module):
    """
    Encoder-style (conformer/ConvBERT-like) hybrid adapted to causal language modelling:
    layer pattern for attn_every=4, 12 layers:  C C C A | C C C A | C C C A
    (C = gated causal depthwise-conv mixer, A = global causal RoPE attention), GeGLU FFN everywhere.
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len,
                 attn_every=4, conv_kernel=15, dropout=0.1, rope_base=10000.0, tie_weights=True, layer_types=None):   
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_heads, max_seq_len, rope_base)
        if layer_types is None: layer_types = [(i + 1) % attn_every == 0 for i in range(n_layers)]      
        self.blocks = nn.ModuleList([
            HybridBlock(d_model, n_heads, d_ff, self.rope, dropout,
                        use_attention=bool(layer_types[i]), conv_kernel=conv_kernel)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run(self, idx, caches=None, offset=0):
        B, T = idx.shape
        if offset + T > self.max_seq_len:
            raise ValueError(f"Sequence length {offset + T} exceeds max_seq_len={self.max_seq_len}")
        x = self.emb_dropout(self.token_emb(idx))
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, cache=None if caches is None else caches[i], offset=offset)
            new_caches.append(c)
        return self.lm_head(self.final_norm(x)), new_caches

    def forward(self, idx, targets=None):
        logits, _ = self._run(idx)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95, eos_id=None, stop_after_eos=None):
        """Prefill the prompt once, then decode token-by-token using conv states + attention KV caches.
        Returns ONLY the newly generated tokens. Batch size 1 for early stopping."""
        self.eval()
        assert idx.size(1) + max_new_tokens <= self.max_seq_len
        logits, caches = self._run(idx)
        P, out, n_eos = idx.size(1), [], 0
        for t in range(max_new_tokens):
            probs = torch.softmax(logits[:, -1, :] / max(temperature, 1e-5), dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            remove = cumulative > top_p
            remove[:, 1:] = remove[:, :-1].clone()
            remove[:, 0] = False
            sorted_probs = sorted_probs.masked_fill(remove, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            nxt = torch.gather(sorted_idx, -1, torch.multinomial(sorted_probs, 1))
            out.append(nxt)
            if stop_after_eos is not None and eos_id is not None and idx.size(0) == 1:
                n_eos += int(nxt.item() == eos_id)
                if n_eos >= stop_after_eos:
                    break
            if t < max_new_tokens - 1:
                logits, caches = self._run(nxt, caches=caches, offset=P + t)
        return torch.cat(out, dim=1)


DEFAULT_LM_CFG = dict(vocab_size=12000, d_model=768, n_layers=12, n_heads=12, d_ff=2560, max_seq_len=1024, attn_every=4,
                      conv_kernel=15, dropout=0.1, rope_base=10000.0, layer_types=None)


def build_lm(cfg):
    c = {**DEFAULT_LM_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_LM_CFG}}
    m = HybridConvAttnLM(vocab_size=c["vocab_size"], d_model=c["d_model"], n_layers=c["n_layers"], n_heads=c["n_heads"], d_ff=c["d_ff"],
                         max_seq_len=c["max_seq_len"], attn_every=c["attn_every"], conv_kernel=c["conv_kernel"], dropout=c["dropout"],
                         rope_base=c["rope_base"], tie_weights=True, layer_types=c["layer_types"])
    return m, c


def load_phase1(path, layer_map=None, dropout=0.1):
    """Load your Phase-1 hybrid (conv + sparse attention).  Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config})
    or a raw state_dict.  Optionally keeps only `layer_map` blocks (KD student; each kept block keeps its own type: conv or attention)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], dict(ck.get("config", {}))
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]; rc = ck.get("run_config", {})
        cfg = {k: v for k, v in dict(n_heads=rc.get("N_HEADS"), attn_every=rc.get("ATTN_EVERY"), conv_kernel=rc.get("CONV_KERNEL"),
                                     max_seq_len=rc.get("BLOCK_SIZE"), rope_base=rc.get("ROPE_BASE")).items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    n = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    types = [f"blocks.{i}.mixer.q_proj.weight" in sd for i in range(n)]               
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    conv = [i for i, t in enumerate(types) if not t]
    if conv: cfg["conv_kernel"] = sd[f"blocks.{conv[0]}.mixer.dw_weight"].shape[-1]
    cfg["dropout"] = dropout
    cfg["n_layers"], cfg["layer_types"] = n, types
    if layer_map is not None:                                
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."): new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"], cfg["layer_types"] = new, len(layer_map), [types[oi] for oi in layer_map]
    lm, cfg = build_lm(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg





def _pick(d, names):
    for n in names:
        if n in d: return d[n]
    return None


def norm_label(v):
    if v is None: return None
    if isinstance(v, (list, tuple)):
        if len(v) != 1: raise ValueError(f"label must be a single 0/1 value, got {v!r}")
        v = v[0]
    if isinstance(v, bool): return int(v)
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ("0", "1"): return int(s)
        raise ValueError(f"unrecognised label {v!r}")
    y = int(v)
    if y not in (0, 1): raise ValueError(f"label must be 0 or 1, got {v!r}")
    return y


def parse_record(rec, key, idx):
    """-> (id, text, label|None).  Accepts {id,text,label} records, or id -> record / id -> text mappings."""
    if isinstance(rec, str): return (key if key is not None else idx), rec, None
    if not isinstance(rec, dict): raise ValueError(f"cannot parse record {key if key is not None else idx}: {type(rec)}")
    rid = _pick(rec, ("id", "case_id", "doc_id", "document_id", "name", "file"))
    if rid is None: rid = key if key is not None else idx
    text = _pick(rec, ("text", "document", "case_text", "content", "sentences"))
    if text is None: raise ValueError(f"record {rid!r} has no text field; keys = {list(rec)[:8]}")
    if isinstance(text, list): text = " ".join(str(x) for x in text)
    return rid, str(text), norm_label(_pick(rec, ("label", "labels", "pred", "judgment", "outcome", "y")))


def read_records(path):
    """Streams a (possibly >1 GB) JSON file that is a list of records or a dict id -> record."""
    try:
        import ijson
    except ImportError:
        ijson = None
    if ijson is None:
        with open(path, encoding="utf-8") as f: obj = json.load(f)
        if isinstance(obj, list):
            for i, r in enumerate(obj): yield parse_record(r, None, i)
        else:
            for i, (k, r) in enumerate(obj.items()): yield parse_record(r, k, i)
        return
    with open(path, "rb") as f:
        first = f.read(64).lstrip()[:1]; f.seek(0)
        if first == b"[":
            for i, r in enumerate(ijson.items(f, "item")): yield parse_record(r, None, i)
        else:
            for i, (k, r) in enumerate(ijson.kvitems(f, "")): yield parse_record(r, k, i)


class WordBPE:
    """Phase-1 encode(): whitespace words -> byte-BPE (min-rank merge loop) -> </s> after every word.  Words are cached."""
    def __init__(self, tok):
        self.tok, self.cache = tok, {}

    def word(self, w):
        r = self.cache.get(w)
        if r is None:
            tok = self.tok
            seq = list(w.encode("utf-8"))
            while len(seq) > 1:
                mergeable = [p for p in zip(seq, seq[1:]) if p in tok.merges]
                if not mergeable: break
                pair = min(mergeable, key=lambda p: tok.merges[p])
                seq = tok._apply_merge(seq, pair, tok.merges[pair])
            r = tuple(seq) + (EOS_ID,)
            self.cache[w] = r
        return r


def tokenize_doc(enc, text, head_words, tail_words, lower=False):
    upper = sum(c.isupper() for c in text[:3000]) / max(1, sum(c.isalpha() for c in text[:3000]))
    if lower: text = text.lower()
    w = text.split(); n = len(w); out = []
    if n <= head_words + tail_words:
        for x in w: out.extend(enc.word(x))
    else:
        for x in w[:head_words]: out.extend(enc.word(x))
        out.append(BOS_ID)
        for x in w[-tail_words:]: out.extend(enc.word(x))
    return np.asarray(out if out else [EOS_ID], dtype=np.uint16), n, upper


def get_docs(args, enc, tok_path):
    key = hashlib.md5(open(tok_path, "rb").read()[:100000] + f"cjpe-v2-{args.head_words}-{args.tail_words}-{args.lower}".encode()).hexdigest()[:10]
    cache = args.cache or os.path.join(args.data_dir, f"cjpecache_{key}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f: out = pickle.load(f)
        print(f"  loaded tokenised cache {cache}")
        return out
    out = {}
    for kind in ("train", "dev", "test"):
        t0 = time.time(); docs = []
        for rid, text, lab in read_records(os.path.join(args.data_dir, f"{kind}.json")):
            toks, nw, up = tokenize_doc(enc, text, args.head_words, args.tail_words, bool(args.lower))
            docs.append(dict(id=rid, y=lab, toks=toks, n_words=nw, upper=up))
            if len(docs) % 5000 == 0: print(f"    {kind}: {len(docs)} cases ({time.time() - t0:.0f}s)", flush=True)
        ids = [d["id"] for d in docs]
        n_dup = len(ids) - len(set(map(str, ids)))
        if kind == "test": assert n_dup == 0, "duplicate ids in test.json (the submission needs exactly one prediction per id)"
        elif n_dup: print(f"  NOTE {kind}.json: {n_dup} repeated ids (single- and multi-instance data are merged); they are kept as separate cases")
        if kind != "test": assert all(d["y"] is not None for d in docs), f"{kind}.json has cases without a label"
        print(f"  {kind}: {len(docs)} cases tokenised in {time.time() - t0:.0f}s")
        out[kind] = docs
    tmp = f"{cache}.{os.getpid()}.tmp"
    with open(tmp, "wb") as f: pickle.dump(out, f)
    os.replace(tmp, cache)
    return out


def print_stats(D_, args):
    for kind, docs in D_.items():
        nw = np.array([d["n_words"] for d in docs]); nt = np.array([len(d["toks"]) for d in docs])
        ys = [d["y"] for d in docs if d["y"] is not None]
        bal = f"label counts 0/1 = {ys.count(0)}/{ys.count(1)} ({100 * np.mean(ys):.1f}% ones)" if ys else "no labels"
        nch = np.minimum(np.ceil(np.maximum(nt - args.window, 0) / max(1, args.stride)) + 1, 99)
        cased = 100 * np.mean([d.get("upper", 0) > 0.02 for d in docs])
        print(f"  {kind:5s} n={len(docs):6d} | {bal} | cased text (>2% capitals): {cased:.1f}% of cases")
        print(f"        words p10/50/90/max = {np.percentile(nw, [10, 50, 90]).astype(int).tolist()}/{nw.max()} | "
              f"kept tokens p10/50/90/max = {np.percentile(nt, [10, 50, 90]).astype(int).tolist()}/{nt.max()} | "
              f"cases longer than one {args.window}-token window: {100 * np.mean(nt > args.window):.1f}%")


def make_chunks(toks, args):
    """Case token array -> list of <= max_chunks arrays (chronological order); the LAST list entry is the end of the case."""
    n, W = len(toks), args.window
    if n <= W: return [toks]
    v = args.view
    if v == "tail": return [toks[-W:]]
    if v == "head": return [toks[:W]]
    if v == "headtail":
        h = min(args.head_tok, W - 1)
        return [np.concatenate([toks[:h], toks[-(W - h):]])]
    ends, e = [], n                                   
    while True:
        ends.append(e)
        if e - W <= 0: break
        e -= args.stride
    if len(ends) > args.max_chunks:
        if args.pick == "tail": ends = ends[:args.max_chunks]
        else:
            idx = np.unique(np.round(np.linspace(0, len(ends) - 1, args.max_chunks)).astype(int)); ends = [ends[i] for i in idx]
    ends = ends[::-1]
    return [toks[max(e, W) - W:max(e, W)] for e in ends]


def collate(docs, idxs, args, device, with_labels=True):
    """-> ids (N,L), lens (N,), chunk_index (B,C) into the N rows (-1 = none), chunk_pos_from_end (B,C), labels (B,)"""
    cls = [make_chunks(docs[i]["toks"], args) for i in idxs]
    flat = [c for cl in cls for c in cl]
    N, L, B, C = len(flat), max(len(c) for c in flat), len(idxs), max(len(cl) for cl in cls)
    ids = torch.full((N, L), PAD_ID, dtype=torch.long); lens = torch.zeros(N, dtype=torch.long)
    for k, c in enumerate(flat):
        ids[k, :len(c)] = torch.from_numpy(c.astype(np.int64)); lens[k] = len(c)
    cidx = torch.full((B, C), -1, dtype=torch.long); cpos = torch.zeros((B, C), dtype=torch.long); off = 0
    for b, cl in enumerate(cls):
        for j in range(len(cl)):
            cidx[b, j] = off + j; cpos[b, j] = min(len(cl) - 1 - j, CHUNK_CAP - 1)
        off += len(cl)
    y = torch.tensor([docs[i]["y"] if (with_labels and docs[i]["y"] is not None) else -100 for i in idxs], dtype=torch.long)
    return ids.to(device), lens.to(device), cidx.to(device), cpos.to(device), y.to(device)


def macro_prf(y_true, y_pred, K=2):
    """Identical to sklearn f1/precision/recall(labels=all K, average='macro', zero_division=0)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cm = np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(1, 2 * tp + fp + fn), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(1, tp + fp), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(1, tp + fn), 0.0)
    return dict(F1=float(f1.mean()), P=float(p.mean()), R=float(r.mean()), per_f1=f1.tolist(), per_p=p.tolist(), per_r=r.tolist(),
                support=cm.sum(1).tolist())


def best_threshold(y, lp):
    """Decision threshold t on s = logp1 - logp0 that maximises dev macro-F1 (ties -> the t closest to 0).  Dev-optimistic."""
    s = lp[:, 1] - lp[:, 0]
    cand = np.unique(np.concatenate([[0.0], np.quantile(s, np.linspace(0.02, 0.98, 97))]))
    best_f, best_t = -1.0, 0.0
    for t in cand:
        f = macro_prf(y, (s > t).astype(int))["F1"]
        if f > best_f + 1e-12 or (abs(f - best_f) <= 1e-12 and abs(t) < abs(best_t)): best_f, best_t = f, float(t)
    return best_t, best_f





def _run_block(b, x):
    return b(x)[0]


class CJPEModel(nn.Module):
    """Phase-1 decoder trunk -> per-chunk vector (mean over real tokens | last real token) -> (+ chunk-from-end embedding)
    -> aggregation over chunks (attn | mean | last) -> dropout -> linear 2-way head."""
    def __init__(self, lm, K=2, pool="last", agg="attn", head_dropout=0.1, grad_ckpt=False):
        super().__init__()
        self.token_emb, self.blocks, self.final_norm = lm.token_emb, lm.blocks, lm.final_norm
        D = self.token_emb.embedding_dim
        self.pool, self.agg, self.grad_ckpt = pool, agg, grad_ckpt
        self.chunk_emb = nn.Embedding(CHUNK_CAP, D); nn.init.normal_(self.chunk_emb.weight, std=0.02)
        self.agg_score = nn.Linear(D, 1); nn.init.zeros_(self.agg_score.weight); nn.init.zeros_(self.agg_score.bias)
        self.drop = nn.Dropout(head_dropout)
        self.head = nn.Linear(D, K)
        self.emb_dropout = lm.emb_dropout

    def forward(self, ids, lens, cidx, cpos):
        N, L = ids.shape
        pad_mask = torch.arange(L, device=ids.device)[None, :] < lens[:, None]            
        x = self.emb_dropout(self.token_emb(ids))             
        for b in self.blocks:
            x = checkpoint(_run_block, b, x, use_reentrant=False) if (self.grad_ckpt and self.training) else b(x)[0]
        h = self.final_norm(x)
        if self.pool == "mean":
            cv = (h.float() * pad_mask.unsqueeze(-1).float()).sum(1) / lens.clamp(min=1).unsqueeze(-1).float()
        else:                                                                               
            cv = h[torch.arange(N, device=ids.device), (lens - 1).clamp(min=0)].float()
        valid = cidx >= 0
        v = cv[cidx.clamp(min=0)] + self.chunk_emb(cpos).float()                           
        vm = valid.unsqueeze(-1).float()
        if self.agg == "mean":
            doc = (v * vm).sum(1) / vm.sum(1).clamp(min=1)
        elif self.agg == "last":
            doc = v[torch.arange(v.size(0), device=v.device), valid.sum(1) - 1]
        else:
            sc = self.agg_score(v).squeeze(-1).float().masked_fill(~valid, float("-inf"))
            doc = (torch.softmax(sc, 1).unsqueeze(-1) * v).sum(1)
        return self.head(self.drop(doc))


class AdapterWrap(nn.Module):
    """Houlsby-style bottleneck adapter applied to the sub-layer output (before the residual add)."""
    def __init__(self, inner, d, bottleneck):
        super().__init__()
        self.inner, self.down, self.up = inner, nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        y = self.inner(*a, **k)
        if isinstance(y, tuple):                                  
            return (y[0] + self.up(F.gelu(self.down(y[0]))),) + tuple(y[1:])
        return y + self.up(F.gelu(self.down(y)))





class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) B A x   (Hu et al. 2021)"""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMAPlusLinear(nn.Module):
    """LoRMA_+ (Bihany et al. 2025): W' = ((alpha/r) B A + I) W0 -- multiplicative, output side.  h = W0 x ; y = h + s B (A h)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))

    def forward(self, x):
        h = self.base(x)
        return h + self.s * F.linear(F.linear(h, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA (Zhang et al. 2023): dW = P diag(E) Q with a rank-r SVD-like triplet structure; triplets are pruned by importance.
    A = Q (r x in), B = P (out x r), E = singular values (r).  `mask` (r) is the pruning state set by AdaLoRACtl."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.s = base, r, alpha / r
        self.E = nn.Parameter(torch.zeros(r))
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.B = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.register_buffer("mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.A) * (self.E * self.mask)
        return self.base(x) + self.s * F.linear(z, self.B)


class AdaLoRACtl:
    """Importance-based rank allocation: sensitivity |w * grad| smoothed with EMA (beta1) times its uncertainty (beta2);
    triplet score = s(E_i) + mean_j s(A_ij) + mean_j s(B_ji).  Global budget follows a cubic schedule from init_r*M to target_r*M."""
    def __init__(self, model, init_r, target_r, tinit, tfinal, delta_t=10, beta1=0.85, beta2=0.85, orth=0.1):
        self.mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.n_total = sum(m.r for m in self.mods)
        self.init_budget, self.target_budget = self.n_total, int(round(target_r / init_r * self.n_total))
        self.tinit, self.tfinal, self.dt, self.b1, self.b2, self.orth = tinit, tfinal, delta_t, beta1, beta2, orth
        self.ipt, self.unc = {}, {}
        self.budget = self.init_budget

    def orth_loss(self):
        if not self.mods or self.orth <= 0: return 0.0
        tot = 0.0
        for m in self.mods:
            A, B = m.A.float(), m.B.float(); I = torch.eye(m.r, device=A.device)
            tot = tot + torch.linalg.matrix_norm(A @ A.T - I) + torch.linalg.matrix_norm(B.T @ B - I)
        return self.orth * tot / len(self.mods)

    def budget_at(self, step, total):
        if step <= self.tinit: return self.init_budget
        if step > total - self.tfinal: return self.target_budget
        c = 1 - (step - self.tinit) / max(1, total - self.tfinal - self.tinit)
        return int((self.init_budget - self.target_budget) * c ** 3 + self.target_budget)

    @torch.no_grad()
    def step(self, step, total):
        """Call after backward (gradients already unscaled), before the optimiser step."""
        final_start = total - self.tfinal
        if step > final_start + 1: return                                              
        for m in self.mods:
            for p in (m.E, m.A, m.B):
                if p.grad is None or not bool(torch.isfinite(p.grad).all()): return    
        for m in self.mods:
            for p in (m.E, m.A, m.B):
                k = id(p); ipt = (p * p.grad).abs()
                a = self.ipt.get(k); a = ipt.clone() if a is None else self.b1 * a + (1 - self.b1) * ipt
                u = self.unc.get(k); u = torch.zeros_like(ipt) if u is None else u
                u = self.b2 * u + (1 - self.b2) * (ipt - a).abs()
                self.ipt[k], self.unc[k] = a, u
        self.budget = self.budget_at(step, total)
        if not (step == final_start + 1 or (step > self.tinit and step % self.dt == 0 and step <= final_start)): return
        scores = []
        for m in self.mods:
            sc = lambda p: self.ipt[id(p)] * self.unc[id(p)]
            scores.append(sc(m.E) + sc(m.A).mean(1) + sc(m.B).mean(0))
        allv = torch.cat(scores)
        k = self.n_total - self.budget
        if k <= 0:
            for m in self.mods: m.mask.fill_(1.0)
            return
        thr = torch.kthvalue(allv, k).values
        for m, sc_ in zip(self.mods, scores): m.mask.copy_((sc_ > thr).float())

    def rank_report(self):
        act = [int(m.mask.sum().item()) for m in self.mods]
        return dict(total_active=sum(act), total_possible=self.n_total, mean_rank=sum(act) / max(1, len(act)))


class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x): return self.base(x * self.l)


ATTN_T = ("q_proj", "k_proj", "v_proj", "in_proj", "out_proj")      
FFN_T = ("gate", "up", "down")
HEAD_PREFIXES = ("head.", "agg_score.", "chunk_emb.")


def is_head_param(name): return name.startswith(HEAD_PREFIXES)


def apply_method(model, args):
    """Freezes the trunk and installs the chosen adaptation.  Returns "pending" for AdaLoRA (controller needs the step count)."""
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for n, p in model.named_parameters():
        if is_head_param(n): p.requires_grad = True                                 
    tg = [t.strip() for t in args.targets.split(",") if t.strip()]
    D = model.token_emb.embedding_dim

    def wrap_linears(factory, names):
        for b in model.blocks:
            for t in names:
                parent = b.mixer if t in ATTN_T else b.ffn
                if isinstance(getattr(parent, t, None), nn.Linear):          
                    setattr(parent, t, factory(getattr(parent, t)))

    ctl = None
    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "partial":                                                          
        for b in model.blocks[-args.top_layers:]:
            for p in b.parameters(): p.requires_grad = True
        for p in model.final_norm.parameters(): p.requires_grad = True
    elif m == "bitfit":                                                           
        for mod in model.modules():
            if isinstance(mod, RMSNorm):
                for p in mod.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "adalora":
        wrap_linears(lambda l: AdaLoRALinear(l, args.ada_init_r, args.alpha_ada, args.lora_dropout), tg)
        ctl = "pending"
    elif m == "adapter":
        for b in model.blocks:
            b.mixer = AdapterWrap(b.mixer, D, args.adapter_dim)
            b.ffn = AdapterWrap(b.ffn, D, args.adapter_dim)
    elif m == "ia3":
        for b in model.blocks:
            for t in [x.strip() for x in args.ia3_targets.split(",") if x.strip()]:
                parent = b.mixer if t in ATTN_T else b.ffn
                lin = getattr(parent, t, None)
                if isinstance(lin, nn.Linear): setattr(parent, t, IA3In(lin) if t == "down" else IA3Out(lin))
    else:
        raise ValueError(m)
    return ctl


def mark_new_params_trainable(model, before_ids):
    for p in model.parameters():
        if id(p) not in before_ids: p.requires_grad = True


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr





@torch.no_grad()
def infer_docs(model, docs, args, device, amp, bs):
    """-> (n, 2) log-probs in the order of `docs`.  Cases are processed shortest-first to minimise padding."""
    was = model.training; model.eval()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["toks"]))
    out = np.zeros((len(docs), 2), np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        ids, lens, cidx, cpos, _ = collate(docs, idx, args, device, with_labels=False)
        with torch.autocast("cuda", enabled=amp):
            logits = model(ids, lens, cidx, cpos)
        out[idx] = F.log_softmax(logits.float(), -1).cpu().numpy()
    if was: model.train()
    return out


def lr_at(step, total, warm, base):
    if step < warm: return base * (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return base * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


VIEW_KEYS = ("view", "window", "max_chunks", "stride", "pick", "head_tok", "head_words", "tail_words", "lower", "pool", "agg")


def meta_of(args, lm_cfg, model):
    d = {k: getattr(args, k) for k in VIEW_KEYS}
    d["pool"] = model.pool                                   
    d.update(method=args.method, lm_cfg=lm_cfg, n_layers=len(model.blocks))
    return d


def load_teacher(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck["meta"]; assert meta["method"] == "full", "KD teacher must be a --method full run (best.pt)"
    lm, _ = build_lm(meta["lm_cfg"])
    model = CJPEModel(lm, 2, pool=meta["pool"], agg=meta["agg"])
    model.load_state_dict(ck["state"]); return model.to(device).eval(), meta


def validate_prediction(out, docs):
    assert len(out) == len(docs) and [o["id"] for o in out] == [d["id"] for d in docs], "prediction ids != case ids (or order differs)"
    assert all(type(o["pred"]) is int and o["pred"] in (0, 1) for o in out), "pred must be a python int, 0 or 1"
    assert len({str(o["id"]) for o in out}) == len(out), "duplicate ids"


def write_prediction(preds, docs, outdir, name="prediction.json"):
    out = [{"id": d["id"], "pred": int(p)} for d, p in zip(docs, preds)]
    validate_prediction(out, docs)
    pj = os.path.join(outdir, name)
    with open(pj, "w", encoding="utf-8") as f: json.dump(out, f)
    with zipfile.ZipFile(os.path.join(outdir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    return pj


def save_lp(path_prefix, docs, lp, with_y=False):
    kw = dict(lp=lp)
    if with_y: kw["y"] = np.array([d["y"] for d in docs])
    np.savez_compressed(path_prefix + "_lp.npz", **kw)
    with open(path_prefix + "_ids.json", "w") as f: json.dump([d["id"] for d in docs], f)


def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep or k.endswith(".mask")}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True, help="folder with train.json dev.json test.json")
    ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--init", required=True, help="Phase-1 weights (model.pt of the hybrid run, or checkpoint.pt)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--method", default="full", choices=["full", "partial", "bitfit", "lora", "lorma", "adalora", "adapter", "ia3", "kd"])
    
    ap.add_argument("--view", default="tail", choices=["tail", "head", "headtail", "chunks"])
    ap.add_argument("--window", type=int, default=1024, help="tokens per chunk (Phase-1 context = RoPE table = 1024, hard maximum)")
    ap.add_argument("--max_chunks", type=int, default=4); ap.add_argument("--stride", type=int, default=1024)
    ap.add_argument("--pick", default="tail", choices=["tail", "spread"], help="which chunks to keep when a case has more than max_chunks")
    ap.add_argument("--head_tok", type=int, default=128, help="head tokens for --view headtail")
    ap.add_argument("--head_words", type=int, default=150); ap.add_argument("--tail_words", type=int, default=1500)
    ap.add_argument("--lower", type=int, default=1, help="lower-case all text before tokenising (1) - makes train/dev/test casing consistent")
    ap.add_argument("--pool", default="auto", choices=["auto", "mean", "last"], help="auto = last real token (model is causal)")
    ap.add_argument("--lm_dropout", type=float, default=0.1, help="dropout inside the trunk during fine-tuning")
    ap.add_argument("--agg", default="attn", choices=["attn", "mean", "last"], help="chunk aggregation (only matters for --view chunks)")
    
    ap.add_argument("--epochs", type=float, default=2.0); ap.add_argument("--bs", type=int, default=4); ap.add_argument("--accum", type=int, default=2)
    ap.add_argument("--lr", type=float, default=None); ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--cw_pow", type=float, default=0.5, help="class weight = (1/freq)^cw_pow, 0 = unweighted")
    ap.add_argument("--grad_ckpt", action="store_true", help="activation checkpointing (use it if --view chunks runs out of GPU memory)")
    ap.add_argument("--tune_thr", type=int, default=1, help="calibrate the decision threshold on dev and apply it to test")
    
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--targets", default="q_proj,k_proj,v_proj,in_proj,out_proj,gate,up,down")
    ap.add_argument("--ada_init_r", type=int, default=24); ap.add_argument("--ada_target_r", type=int, default=16)
    ap.add_argument("--alpha_ada", type=float, default=48); ap.add_argument("--ada_orth", type=float, default=0.1)
    ap.add_argument("--ada_tinit_frac", type=float, default=0.1); ap.add_argument("--ada_tfinal_frac", type=float, default=0.3)
    ap.add_argument("--ia3_targets", default="k_proj,v_proj,in_proj,down")
    ap.add_argument("--adapter_dim", type=int, default=64); ap.add_argument("--top_layers", type=int, default=4)
    ap.add_argument("--teacher", default=None); ap.add_argument("--student_layers", type=int, default=6)
    ap.add_argument("--kd_T", type=float, default=2.0); ap.add_argument("--kd_alpha", type=float, default=0.5)
    
    ap.add_argument("--time_budget_min", type=float, default=75.0)
    ap.add_argument("--eval_per_epoch", type=int, default=2)
    ap.add_argument("--max_train_docs", type=int, default=0); ap.add_argument("--max_dev_docs", type=int, default=0)
    ap.add_argument("--no_test", action="store_true"); ap.add_argument("--prepare_only", action="store_true")
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--cache", default=None)
    args = ap.parse_args()

    t_global = time.time()
    os.makedirs(args.out, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    default_lr = dict(full=5e-5, kd=1e-4, partial=1e-4, bitfit=2e-3, lora=1e-3, lorma=1e-3, adalora=1e-3, adapter=1e-3, ia3=1e-2)   
    base_lr = args.lr if args.lr is not None else default_lr[args.method]
    assert 1 <= args.max_chunks <= CHUNK_CAP

    
    tok = ByteLevelBPETokenizer.load(args.tokenizer)
    enc = WordBPE(tok)
    teacher = None; tmeta = None
    if args.method == "kd":
        assert args.teacher, "--teacher runs/<full_run>/best.pt is required for kd"
        teacher, tmeta = load_teacher(args.teacher, device)
        for k in VIEW_KEYS: setattr(args, k, tmeta[k])                       
    D_ = get_docs(args, enc, args.tokenizer)
    print_stats(D_, args)
    if args.prepare_only: return
    train_docs, dev_docs, test_docs = D_["train"], D_["dev"], D_["test"]
    rs = random.Random(args.seed)
    if args.max_train_docs and len(train_docs) > args.max_train_docs: train_docs = rs.sample(train_docs, args.max_train_docs)
    if args.max_dev_docs and len(dev_docs) > args.max_dev_docs: dev_docs = rs.sample(dev_docs, args.max_dev_docs)

    
    if args.method == "kd":
        layer_map = [int(round(x)) for x in np.linspace(0, tmeta["lm_cfg"]["n_layers"] - 1, args.student_layers)]
        print(f"  KD student = teacher blocks {layer_map}; teacher params={count_params(teacher)[0] / 1e6:.1f}M")
        lm, lm_cfg = load_phase1(args.init, layer_map=layer_map, dropout=args.lm_dropout)
    else:
        lm, lm_cfg = load_phase1(args.init, dropout=args.lm_dropout)
    pool = args.pool if args.pool != "auto" else "last"
    assert args.window <= lm_cfg["max_seq_len"], f"window {args.window} longer than the Phase-1 context / RoPE table ({lm_cfg['max_seq_len']})"
    model = CJPEModel(lm, 2, pool=pool, agg=args.agg, grad_ckpt=args.grad_ckpt)
    if args.method == "kd":
        with torch.no_grad():
            for nm in ("head", "agg_score", "chunk_emb"): getattr(model, nm).load_state_dict(getattr(teacher, nm).state_dict())
    before = {id(p) for p in model.parameters()}
    ctl = apply_method(model, args)
    mark_new_params_trainable(model, before)
    model.to(device)
    tot, trn = count_params(model)
    print(f"== method={args.method} view={args.view} pool={pool} agg={args.agg} lr={base_lr} lr_head={args.lr_head} device={device}")
    print(f"  params total={tot / 1e6:.2f}M trainable={trn / 1e6:.3f}M ({100 * trn / tot:.2f}%) | train={len(train_docs)} dev={len(dev_docs)} test={len(test_docs)}")

    cnt = np.bincount([d["y"] for d in train_docs], minlength=2).astype(float)
    cw = np.ones(2)
    if args.cw_pow > 0:
        fr = np.maximum(cnt, 1) / cnt.sum(); cw = (1.0 / fr) ** args.cw_pow; cw = cw / (cw * fr).sum()
    cw_t = torch.tensor(cw, dtype=torch.float, device=device)
    print(f"  train label counts {cnt.astype(int).tolist()} -> class weights {np.round(cw, 3).tolist()}")

    decay, no_decay, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (head_p if is_head_param(n) else decay if p.ndim >= 2 else no_decay).append(p)
    groups = [dict(params=head_p, lr=args.lr_head, weight_decay=0.0), dict(params=no_decay, lr=base_lr, weight_decay=0.0),
              dict(params=decay, lr=base_lr, weight_decay=args.wd)]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    rng = random.Random(args.seed)
    meta = meta_of(args, lm_cfg, model)
    per = args.bs * args.accum
    assert len(train_docs) >= per, f"need at least bs*accum={per} training cases"
    steps_per_epoch = max(1, len(train_docs) // per)
    planned = max(1, int(args.epochs * steps_per_epoch))
    total, step, best_key, best_step, t_bench, best_state = planned, 0, -1.0, 0, None, None
    eval_every = max(1, steps_per_epoch // max(1, args.eval_per_epoch))
    print(f"  steps/epoch={steps_per_epoch} (effective batch {per} cases) planned steps={planned} eval every {eval_every}")
    if args.method == "adalora":
        ctl = AdaLoRACtl(model, args.ada_init_r, args.ada_target_r, tinit=max(40, int(args.ada_tinit_frac * planned)),
                         tfinal=int(args.ada_tfinal_frac * planned), orth=args.ada_orth)
        print(f"  AdaLoRA: {len(ctl.mods)} modules, budget {ctl.init_budget} -> {ctl.target_budget} triplets, tinit={ctl.tinit} tfinal={ctl.tfinal}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    y_dev = np.array([d["y"] for d in dev_docs])

    def run_eval(tag):
        nonlocal best_key, best_step, best_state
        lp = infer_docs(model, dev_docs, args, device, amp, args.bs * 2)
        f_arg = macro_prf(y_dev, lp.argmax(1))["F1"]; thr, f_thr = best_threshold(y_dev, lp)
        key = f_thr if args.tune_thr else f_arg
        print(f"  [eval {tag}] step={step} dev macro-F1 argmax={f_arg:.4f} | calibrated={f_thr:.4f} (thr={thr:+.3f})", flush=True)
        if key > best_key: best_key, best_step, best_state = key, step, trainable_state(model)
        model.train()

    model.train()
    t_start = time.time(); done = False; ep = 0; loss_ema = None
    while not done:
        order = list(range(len(train_docs))); rng.shuffle(order); ep += 1
        for bi in range(0, len(order) - per + 1, per):
            if step == 5: t_bench = time.time()
            if step == 35 and t_bench is not None:                      
                sps = (time.time() - t_bench) / 30
                left = args.time_budget_min * 60 * 0.80 - (time.time() - t_global)
                total = max(60, min(planned, 35 + int(left / sps)))
                print(f"  speed={1 / sps:.2f} steps/s -> total steps={total} (planned {planned})", flush=True)
            if step >= total or (time.time() - t_global) > args.time_budget_min * 60 * 0.95: done = True; break
            lr_scale = lr_at(step, total, args.warmup, 1.0)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_scale
            opt.zero_grad(set_to_none=True); loss_val = 0.0
            for a in range(args.accum):
                idx = order[bi + a * args.bs: bi + (a + 1) * args.bs]
                ids, lens, cidx, cpos, y = collate(train_docs, idx, args, device)
                with torch.autocast("cuda", enabled=amp):
                    logits = model(ids, lens, cidx, cpos)
                lf = logits.float()
                loss = F.cross_entropy(lf, y, weight=cw_t)
                if teacher is not None:
                    with torch.no_grad(), torch.autocast("cuda", enabled=amp):
                        tl = teacher(ids, lens, cidx, cpos)
                    T = args.kd_T
                    kd = F.kl_div(F.log_softmax(lf / T, -1), F.softmax(tl.float() / T, -1), reduction="batchmean") * T * T
                    loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
                if ctl is not None: loss = loss + ctl.orth_loss()
                scaler.scale(loss / args.accum).backward(); loss_val += loss.item() / args.accum
            scaler.unscale_(opt)
            if ctl is not None: ctl.step(step + 1, total)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            loss_ema = loss_val if loss_ema is None else 0.98 * loss_ema + 0.02 * loss_val
            if step % 25 == 0:
                print(f"  step {step}/{total} ep {ep} loss {loss_ema:.4f} lr {base_lr * lr_scale:.2e} {(time.time() - t_start) / 60:.1f}min", flush=True)
            if step % eval_every == 0 and step < total and step >= 35: run_eval("mid")
    train_min = (time.time() - t_start) / 60
    run_eval("final")
    peak_gb = torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0

    
    model.load_state_dict(best_state, strict=False)
    if args.method == "full":
        torch.save({"state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "meta": meta}, os.path.join(args.out, "best.pt"))
    else:
        torch.save(best_state, os.path.join(args.out, "trainable.pt"))
    t0 = time.time()
    dev_lp = infer_docs(model, dev_docs, args, device, amp, args.bs * 2)
    n_tok = sum(sum(len(c) for c in make_chunks(d["toks"], args)) for d in dev_docs); inf_tps = n_tok / max(1e-6, time.time() - t0)
    save_lp(os.path.join(args.out, "dev"), dev_docs, dev_lp, with_y=True)
    sc_arg = macro_prf(y_dev, dev_lp.argmax(1)); thr, f_thr = best_threshold(y_dev, dev_lp)
    thr_used = thr if args.tune_thr else 0.0
    sc = macro_prf(y_dev, ((dev_lp[:, 1] - dev_lp[:, 0]) > thr_used).astype(int))
    print(f"  dev macro-F1 argmax={sc_arg['F1']:.4f} | with threshold {thr_used:+.3f}: F1={sc['F1']:.4f} P={sc['P']:.4f} R={sc['R']:.4f}")
    print(f"  per-class F1 (0/1) = {np.round(sc['per_f1'], 3).tolist()}  support = {sc['support']}")
    with open(os.path.join(args.out, "dev_pred.json"), "w") as f:
        json.dump([{"id": d["id"], "pred": int(p)} for d, p in zip(dev_docs, ((dev_lp[:, 1] - dev_lp[:, 0]) > thr_used).astype(int))], f)

    res = dict(method=args.method, view=args.view, attn="causal-hybrid", pool=pool, agg=args.agg, params_total=tot, params_trainable=trn,
               train_minutes=train_min, steps=step, planned_steps=planned, peak_gpu_gb=peak_gb, infer_tokens_per_s=inf_tps, best_step=best_step,
               dev_macro_F1=sc["F1"], dev_macro_F1_argmax=sc_arg["F1"], dev_macro_P=sc["P"], dev_macro_R=sc["R"], threshold=thr_used,
               dev_per_class_F1=sc["per_f1"], dev_support=sc["support"], args=vars(args), n_dev=len(dev_docs))
    if ctl is not None and args.method == "adalora": res["adalora_rank"] = ctl.rank_report(); print("  AdaLoRA final ranks:", res["adalora_rank"])
    if not args.no_test:
        test_lp = infer_docs(model, test_docs, args, device, amp, args.bs * 2)
        save_lp(os.path.join(args.out, "test"), test_docs, test_lp)
        preds = ((test_lp[:, 1] - test_lp[:, 0]) > thr_used).astype(int)
        write_prediction(preds, test_docs, args.out)
        res["n_test"], res["test_pred_ones_frac"] = len(test_docs), float(preds.mean())
        print(f"  wrote {args.out}/submission.zip ({len(test_docs)} cases, {100 * preds.mean():.1f}% predicted 1)")
    with open(os.path.join(args.out, "results.json"), "w") as f: json.dump(res, f, indent=1, default=str)
    print(f"== done in {(time.time() - t_global) / 60:.1f} min | dev macro-F1 = {res['dev_macro_F1']:.4f}")


if __name__ == "__main__":
    main()


Writing cjpe_hybrid.py


In [4]:
%%writefile ensemble4_hybrid.py

"""
ensemble4_hybrid.py -- combine Phase-4 runs (each run dir holds dev_lp.npz/dev_ids.json/test_lp.npz/test_ids.json written by cjpe.py).

  python ensemble4_hybrid.py --runs_dir runs --names full,lorma,adalora --out subs/ens_a
  python ensemble4_hybrid.py --runs_dir runs --all --greedy --out subs/ens_greedy          # forward selection on dev

The averaged class probabilities get their own dev-calibrated decision threshold.  Anything selected on dev (greedy subset,
threshold) is optimistically scored on dev -- the printed number is NOT an unbiased estimate; the Codabench score is.
An ensemble is an EXTRA submission: it does not count as a separate fine-tuning technique.
"""
import argparse, glob, json, os, sys
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import cjpe_hybrid as cjpe


def load_run(d, kind):
    z = np.load(os.path.join(d, f"{kind}_lp.npz"), allow_pickle=False)
    ids = json.load(open(os.path.join(d, f"{kind}_ids.json")))
    return ids, z["lp"], (z["y"] if "y" in z.files else None)


def combine(lps):
    p = np.mean([np.exp(l.astype(np.float64)) for l in lps], 0)
    return np.log(np.maximum(p, 1e-12))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs_dir", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--names", default=""); ap.add_argument("--all", action="store_true"); ap.add_argument("--greedy", action="store_true")
    a = ap.parse_args()
    names = [n for n in a.names.split(",") if n]
    if a.all:
        names = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(a.runs_dir, "*", "test_lp.npz"))
                       if not os.path.basename(os.path.dirname(p)).startswith("smoke"))
    assert names, "give --names a,b,c or --all"
    dev, test = {}, {}
    for n in names:
        d = os.path.join(a.runs_dir, n); dev[n] = load_run(d, "dev"); test[n] = load_run(d, "test")
    ref = names[0]
    for n in names:
        assert dev[n][0] == dev[ref][0] and test[n][0] == test[ref][0], f"{n}: case ids/order differ from {ref} (runs must use the same dev/test files and no --max_*_docs)"
    y = dev[ref][2]

    def score(sel):
        lp = combine([dev[n][1] for n in sel]); t, f = cjpe.best_threshold(y, lp); return f, t
    sel = names
    if a.greedy:
        sel, best = [], -1.0
        while True:
            cand = [(score(sel + [n])[0], n) for n in names if n not in sel]
            if not cand: break
            f, n = max(cand)
            if f <= best + 1e-9: break
            sel, best = sel + [n], f
            print(f"  + {n:12s} -> dev macro-F1 {f:.4f}")
    f, t = score(sel)
    print(f"ensemble of {sel}: dev macro-F1 {f:.4f} at threshold {t:+.3f}")
    lp_t = combine([test[n][1] for n in sel]); preds = ((lp_t[:, 1] - lp_t[:, 0]) > t).astype(int)
    os.makedirs(a.out, exist_ok=True)
    cjpe.write_prediction(preds, [{"id": i} for i in test[ref][0]], a.out)
    print(f"wrote {a.out}/submission.zip ({len(preds)} cases, {100 * preds.mean():.1f}% predicted 1)")


if __name__ == "__main__":
    main()


Writing ensemble4_hybrid.py


### 4. Tokenise once (single process) and look at the data
Tokenise only the first `head_words` and last `tail_words` words of each case. The cache is shared by both GPUs; run this once before the parallel jobs.
Use the printed class balance and length statistics to choose class weights and chunk count. Note that your tokenizer emits one `</s>` token per word, so 512 tokens cover only ≈ 150–250 words.

**Check the `cased text` percentages.** The default `--lower 1` normalises casing across splits. Compare raw casing with `--lower 0` (which builds a separate cache).

In [5]:
!python cjpe_hybrid.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {WORK}/runs/prep --prepare_only

    train: 5000 cases (8s)
    train: 10000 cases (13s)
    train: 15000 cases (19s)
    train: 20000 cases (25s)
    train: 25000 cases (31s)
    train: 30000 cases (36s)
    train: 35000 cases (41s)
  NOTE train.json: 5082 repeated ids (single- and multi-instance data are merged); they are kept as separate cases
  train: 37387 cases tokenised in 43s
  NOTE dev.json: 994 repeated ids (single- and multi-instance data are merged); they are kept as separate cases
  dev: 1988 cases tokenised in 2s
  test: 1517 cases tokenised in 2s
  train n= 37387 | label counts 0/1 = 22067/15320 (41.0% ones) | cased text (>2% capitals): 98.7% of cases
        words p10/50/90/max = [408, 2192, 6575]/412606 | kept tokens p10/50/90/max = [906, 3614, 3781]/5828 | cases longer than one 1024-token window: 88.7%
  dev   n=  1988 | label counts 0/1 = 994/994 (50.0% ones) | cased text (>2% capitals): 0.0% of cases
        words p10/50/90/max = [1200, 2821, 6933]/42198 | kept tokens p10/50/90/max = [2688, 3651, 3

In [6]:
os.makedirs(f"{WORK}/runs", exist_ok=True)
BUDGET = 55   

def cmd(name, gpu, extra, budget=None, teacher_run=None):
    out = f"{WORK}/runs/{name}"; os.makedirs(out, exist_ok=True)
    t = f"--teacher {WORK}/runs/{teacher_run}/best.pt" if teacher_run else ""
    return (f"PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} python cjpe_hybrid.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {out} "
            f"--time_budget_min {budget or BUDGET} {t} {extra} > {out}/log.txt 2>&1")

def launch(gpu, jobs):
    """jobs = [(name, extra_args, teacher_run_or_None)] executed one after another on `gpu`."""
    p = subprocess.Popen(" ; ".join(cmd(n, gpu, e, teacher_run=t) for n, e, t in jobs), shell=True)
    return p, [j[0] for j in jobs]

def wait(procs, every=120):
    while any(p.poll() is None for p, _ in procs):
        time.sleep(every)
        for p, names in procs:
            logs = [l for n in names for l in glob.glob(f"{WORK}/runs/{n}/log.txt")]
            if not logs: continue
            lg = max(logs, key=os.path.getmtime); tail = open(lg).read().strip().splitlines()[-1:]
            print(f"[{lg.split('/')[-2]}] {'running' if p.poll() is None else 'done'} | {tail[0][:130] if tail else ''}")
    for p, names in procs:
        for n in names:
            print(f"== {n}: {'OK' if os.path.exists(f'{WORK}/runs/{n}/results.json') else 'FAILED  <-- see runs/' + n + '/log.txt'}")

def table(pattern="*"):
    rows = []
    for d in sorted(glob.glob(f"{WORK}/runs/{pattern}/results.json")):
        x = json.load(open(d))
        rows.append(dict(run=d.split("/")[-2], method=x["method"], view=x["view"], attn=x["attn"], dev_F1=round(x["dev_macro_F1"], 4),
                         F1_argmax=round(x["dev_macro_F1_argmax"], 4), P=round(x["dev_macro_P"], 4), R=round(x["dev_macro_R"], 4), thr=round(x["threshold"], 3),
                         trainable_M=round(x["params_trainable"] / 1e6, 3), total_M=round(x["params_total"] / 1e6, 1), train_min=round(x["train_minutes"], 1),
                         steps=f"{x['steps']}/{x['planned_steps']}", gpu_GB=round(x["peak_gpu_gb"], 1), infer_tok_s=int(x["infer_tokens_per_s"])))
    return pd.DataFrame(rows).sort_values("dev_F1", ascending=False) if rows else pd.DataFrame()

def collect():
    """Copy every finished run's submission.zip to /kaggle/working/submissions/<run>.zip and verify it. Call after each round."""
    import zipfile, hashlib
    os.makedirs(f"{WORK}/submissions", exist_ok=True)
    test_ids = None
    test_path = f"{DATA}/test.json"
    if os.path.exists(test_path):
        t = json.load(open(test_path))
        if isinstance(t, list): test_ids = [str(r["id"]) for r in t]
    for d in sorted(glob.glob(f"{WORK}/runs/*/submission.zip")):
        n = d.split("/")[-2]
        if n.startswith("smoke") or n == "prep": continue
        shutil.copy(d, f"{WORK}/submissions/{n}.zip")
    for z in sorted(glob.glob(f"{WORK}/submissions/*.zip")):
        zf = zipfile.ZipFile(z); p = json.load(zf.open("prediction.json"))
        ok = (test_ids is None) or ([str(x["id"]) for x in p] == test_ids)
        print(f"{os.path.basename(z):24s} root_files={zf.namelist()} n={len(p)} ones={100*np.mean([x['pred'] for x in p]):.1f}% ids_match_test={ok} md5={hashlib.md5(zf.read('prediction.json')).hexdigest()[:8]}")

def move_smoke():
    os.makedirs(f"{WORK}/smoke", exist_ok=True)
    for d in glob.glob(f"{WORK}/runs/smoke_*"): shutil.move(d, f"{WORK}/smoke/" + os.path.basename(d))

### 6. Smoke test (≈ 5–10 min). Every method must say OK before you spend GPU hours.

In [7]:
SMOKE = "--max_train_docs 64 --max_dev_docs 32 --epochs 1 --no_test --bs 2 --accum 2"
pA = launch(0, [("smoke_full", f"--method full {SMOKE}", None), ("smoke_kd", f"--method kd {SMOKE}", "smoke_full"), ("smoke_lora", f"--method lora {SMOKE}", None),
                ("smoke_adalora", f"--method adalora {SMOKE}", None), ("smoke_chunks", f"--method lorma --view chunks --max_chunks 4 --grad_ckpt {SMOKE}", None)])
pB = launch(1, [("smoke_lorma", f"--method lorma {SMOKE}", None), ("smoke_bitfit", f"--method bitfit {SMOKE}", None), ("smoke_adapter", f"--method adapter {SMOKE}", None),
                ("smoke_partial", f"--method partial {SMOKE}", None), ("smoke_ia3", f"--method ia3 {SMOKE}", None)])
wait([pA, pB], every=30)
display(table("smoke_*")); print(open(f"{WORK}/runs/smoke_chunks/log.txt").read()[-1500:])
move_smoke()

[smoke_kd] running |   steps/epoch=16 (effective batch 4 cases) planned steps=16 eval every 8
[smoke_bitfit] running |   steps/epoch=16 (effective batch 4 cases) planned steps=16 eval every 8
[smoke_adalora] running |   AdaLoRA: 66 modules, budget 1584 -> 1056 triplets, tinit=40 tfinal=4
[smoke_ia3] running |         words p10/50/90/max = [1212, 2940, 7302]/40487 | kept tokens p10/50/90/max = [2731, 3654, 3786]/4190 | cases longer than o
[smoke_chunks] running |   steps/epoch=16 (effective batch 4 cases) planned steps=16 eval every 8
[smoke_ia3] done | == done in 0.2 min | dev macro-F1 = 0.6532
[smoke_chunks] done | == done in 0.5 min | dev macro-F1 = 0.5883
[smoke_ia3] done | == done in 0.2 min | dev macro-F1 = 0.6532
== smoke_full: OK
== smoke_kd: OK
== smoke_lora: OK
== smoke_adalora: OK
== smoke_chunks: OK
== smoke_lorma: OK
== smoke_bitfit: OK
== smoke_adapter: OK
== smoke_partial: OK
== smoke_ia3: OK


,run,method,view,attn,dev_F1,F1_argmax,P,R,thr,trainable_M,total_M,train_min,steps,gpu_GB,infer_tok_s
0,smoke_adalora,adalora,tail,causal-hybrid,0.6532,0.5934,0.6548,0.6529,-0.028,4.163,107.3,0.1,16/16,2.0,46134
1,smoke_adapter,adapter,tail,causal-hybrid,0.6532,0.6532,0.6548,0.6529,0.000,2.394,105.5,0.1,16/16,1.6,62386
2,smoke_bitfit,bitfit,tail,causal-hybrid,0.6532,0.6235,0.6548,0.6529,-0.004,0.034,103.1,0.1,16/16,1.5,66496
4,smoke_full,full,tail,causal-hybrid,0.6532,0.6532,0.6548,0.6529,0.000,103.135,103.1,0.1,16/16,2.9,66830
5,smoke_ia3,ia3,tail,causal-hybrid,0.6532,0.6235,0.6548,0.6529,-0.006,0.064,103.2,0.1,16/16,1.8,54101
9,smoke_partial,partial,tail,causal-hybrid,0.6532,0.6532,0.6548,0.6529,0.000,31.317,103.1,0.1,16/16,1.2,65590
8,smoke_lorma,lorma,tail,causal-hybrid,0.6532,0.6532,0.6548,0.6529,0.000,3.234,106.4,0.1,16/16,1.9,47891
7,smoke_lora,lora,tail,causal-hybrid,0.6250,0.5934,0.6275,0.6275,0.032,2.779,105.9,0.1,16/16,2.0,46543
6,smoke_kd,kd,tail,causal-hybrid,0.6190,0.4980,0.6235,0.6196,-0.311,56.473,56.5,0.1,16/16,2.2,131807
3,smoke_chunks,lorma,chunks,causal-hybrid,0.5883,0.4375,0.8036,0.6333,-0.407,3.234,106.4,0.3,16/16,1.3,49762


  loaded tokenised cache /kaggle/working/p4data/cjpecache_f9c407245d.pkl
  train n= 37387 | label counts 0/1 = 22067/15320 (41.0% ones) | cased text (>2% capitals): 98.7% of cases
        words p10/50/90/max = [408, 2192, 6575]/412606 | kept tokens p10/50/90/max = [906, 3614, 3781]/5828 | cases longer than one 1024-token window: 88.7%
  dev   n=  1988 | label counts 0/1 = 994/994 (50.0% ones) | cased text (>2% capitals): 0.0% of cases
        words p10/50/90/max = [1200, 2821, 6933]/42198 | kept tokens p10/50/90/max = [2688, 3651, 3792]/4135 | cases longer than one 1024-token window: 99.0%
  test  n=  1517 | label counts 0/1 = 755/762 (50.2% ones) | cased text (>2% capitals): 1.1% of cases
        words p10/50/90/max = [1212, 2940, 7302]/40487 | kept tokens p10/50/90/max = [2731, 3654, 3786]/4190 | cases longer than one 1024-token window: 98.4%
== method=lorma view=chunks pool=last agg=attn lr=0.001 lr_head=0.001 device=cuda
  params total=106.35M trainable=3.234M (3.04%) | train=64 de

### 7. Experiment configuration — **your decisions**
These Phase 2/3 defaults are starting points. Tune them using dev results and record your reasoning.
* `VIEW` — how a case is shown to the model: `tail` (last 512 tokens), `head`, `headtail`, or `chunks` (up to `max_chunks` windows + attention pooling).
* The hybrid trunk (causal convs + causal attention) uses the last real token as the case vector (`--pool last`, default); `--pool mean` is available. Context is 1024 tokens (hard maximum).
* Spec §10.2: ≥ 5 *different adaptation techniques* (LoRMA mandatory) + 1 knowledge-distillation run. Changing only view, rank, or learning rate does **not** count as a new technique.

In [8]:
EP   = dict(full=2, partial=2, bitfit=3, lora=3, lorma=3, adalora=3, adapter=3, ia3=4, kd=3)    
RANK = 16
VIEW = "--view tail --window 1024"                     
CHUNKS = "--view chunks --window 1024 --stride 1024 --max_chunks 3 --pick tail --agg attn --grad_ckpt --bs 2 --accum 4"
COMMON = "--cw_pow 0.5 --bs 4 --accum 2"
def X(m, view=VIEW, extra=""):
    return f"--method {m} --epochs {EP[m]} {COMMON} {view} {extra}"


### 8. Round 1 — full fine-tuning (also the KD teacher) ‖ LoRMA

In [9]:
pA = launch(0, [("full",  X("full"), None)])
pB = launch(1, [("lorma", X("lorma", extra=f"--rank {RANK} --alpha {2*RANK}"), None)])
wait([pA, pB]); display(table()); collect()

[full] running |   step 250/6035 ep 1 loss 0.7194 lr 4.99e-05 1.9min
[lorma] running |   step 225/6046 ep 1 loss 0.8354 lr 9.99e-04 1.7min
[full] running |   step 500/6035 ep 1 loss 0.7438 lr 4.94e-05 3.8min
[lorma] running |   step 500/6046 ep 1 loss 0.7837 lr 9.89e-04 3.8min
[full] running |   step 750/6035 ep 1 loss 0.6646 lr 4.85e-05 5.7min
[lorma] running |   step 775/6046 ep 1 loss 0.7786 lr 9.69e-04 5.9min
[full] running |   step 1025/6035 ep 1 loss 0.6163 lr 4.71e-05 7.9min
[lorma] running |   step 1050/6046 ep 1 loss nan lr 9.38e-04 7.9min
[full] running |   step 1275/6035 ep 1 loss 0.5998 lr 4.53e-05 9.8min
[lorma] running |   step 1300/6046 ep 1 loss nan lr 9.03e-04 9.7min
[full] running |   step 1525/6035 ep 1 loss 0.6205 lr 4.32e-05 11.8min
[lorma] running |   step 1575/6046 ep 1 loss nan lr 8.56e-04 11.7min
[full] running |   step 1800/6035 ep 1 loss 0.6092 lr 4.06e-05 13.9min
[lorma] running |   step 1850/6046 ep 1 loss nan lr 8.01e-04 13.8min
[full] running |   step 205

,run,method,view,attn,dev_F1,F1_argmax,P,R,thr,trainable_M,total_M,train_min,steps,gpu_GB,infer_tok_s
0,full,full,tail,causal-hybrid,0.7058,0.6787,0.7153,0.7082,-0.617,103.135,103.1,48.6,6035/9346,4.1,57918
1,lorma,lorma,tail,causal-hybrid,0.4645,0.3798,0.4812,0.4839,-0.063,3.234,106.4,46.1,6046/14019,3.1,45496


full.zip                 root_files=['prediction.json'] n=1517 ones=41.9% ids_match_test=True md5=f8052bcc
lorma.zip                root_files=['prediction.json'] n=1517 ones=31.9% ids_match_test=True md5=1750fed5


### 9. Round 2 — AdaLoRA ‖ BitFit-style (RMSNorm gains)

In [10]:
pA = launch(0, [("adalora", X("adalora", extra="--ada_init_r 24 --ada_target_r 16 --alpha_ada 48"), None)])
pB = launch(1, [("bitfit",  X("bitfit"), None)])
wait([pA, pB]); display(table()); collect()

[adalora] running |   step 175/4707 ep 1 loss 0.8047 lr 9.99e-04 1.7min
[bitfit] running |   step 325/8082 ep 1 loss 0.7426 lr 2.00e-03 1.9min
[adalora] running |   step 400/4707 ep 1 loss 0.7308 lr 9.90e-04 3.9min
[bitfit] running |   step 650/8082 ep 1 loss 0.7017 lr 1.98e-03 3.8min
[adalora] running |   step 600/4707 ep 1 loss 0.6849 lr 9.71e-04 5.7min
[bitfit] running |   step 1000/8082 ep 1 loss 0.6857 lr 1.94e-03 5.8min
[adalora] running |   step 825/4707 ep 1 loss 0.6893 lr 9.40e-04 7.8min
[bitfit] running |   step 1350/8082 ep 1 loss 0.7089 lr 1.88e-03 7.9min
[adalora] running |   step 1050/4707 ep 1 loss 0.6474 lr 8.99e-04 9.8min
[bitfit] running |   step 1675/8082 ep 1 loss 0.6545 lr 1.81e-03 9.8min
[adalora] running |   step 1275/4707 ep 1 loss 0.6743 lr 8.48e-04 11.9min
[bitfit] running |   step 2025/8082 ep 1 loss 0.6840 lr 1.73e-03 11.8min
[adalora] running |   step 1475/4707 ep 1 loss 0.6982 lr 7.96e-04 13.7min
[bitfit] running |   step 2325/8082 ep 1 loss 0.6283 lr 1.64

,run,method,view,attn,dev_F1,F1_argmax,P,R,thr,trainable_M,total_M,train_min,steps,gpu_GB,infer_tok_s
2,full,full,tail,causal-hybrid,0.7058,0.6787,0.7153,0.7082,-0.617,103.135,103.1,48.6,6035/9346,4.1,57918
0,adalora,adalora,tail,causal-hybrid,0.6679,0.5905,0.6683,0.6680,-0.711,4.163,107.3,44.7,4707/14019,3.3,42618
1,bitfit,bitfit,tail,causal-hybrid,0.6245,0.5322,0.6402,0.6308,-0.748,0.034,103.1,49.0,8082/14019,2.3,60873
3,lorma,lorma,tail,causal-hybrid,0.4645,0.3798,0.4812,0.4839,-0.063,3.234,106.4,46.1,6046/14019,3.1,45496


adalora.zip              root_files=['prediction.json'] n=1517 ones=54.1% ids_match_test=True md5=ee4d45f0
bitfit.zip               root_files=['prediction.json'] n=1517 ones=40.2% ids_match_test=True md5=b577ff98
full.zip                 root_files=['prediction.json'] n=1517 ones=41.9% ids_match_test=True md5=f8052bcc
lorma.zip                root_files=['prediction.json'] n=1517 ones=31.9% ids_match_test=True md5=1750fed5


### 10. Round 3 — Houlsby-style adapters ‖ partial fine-tuning (top-4 blocks)

In [11]:
pA = launch(0, [("adapter", X("adapter", extra="--adapter_dim 64"), None)])
pB = launch(1, [("partial", X("partial", extra="--top_layers 4"), None)])
wait([pA, pB]); display(table()); collect()

[adapter] running |   step 300/7927 ep 1 loss 0.7549 lr 9.98e-04 1.8min
[partial] running |   step 450/9346 ep 1 loss 0.7465 lr 9.96e-05 1.9min
[adapter] running |   step 625/7927 ep 1 loss 0.6748 lr 9.89e-04 3.8min
[partial] running |   step 925/9346 ep 1 loss 0.6466 lr 9.81e-05 3.9min
[adapter] running |   step 975/7927 ep 1 loss 0.6734 lr 9.70e-04 5.9min
[partial] running |   step 1400/9346 ep 1 loss 0.6881 lr 9.52e-05 5.8min
[adapter] running |   step 1300/7927 ep 1 loss 0.6702 lr 9.43e-04 7.9min
[partial] running |   step 1875/9346 ep 1 loss 0.6183 lr 9.12e-05 7.8min
[adapter] running |   step 1625/7927 ep 1 loss 0.6476 lr 9.09e-04 9.8min
[partial] running |   step 2325/9346 ep 1 loss 0.6284 lr 8.64e-05 9.7min
[adapter] running |   step 1950/7927 ep 1 loss 0.6493 lr 8.69e-04 11.8min
[partial] running |   step 2725/9346 ep 1 loss 0.6392 lr 8.14e-05 11.9min
[adapter] running |   step 2275/7927 ep 1 loss 0.6574 lr 8.21e-04 13.8min
[partial] running |   step 3200/9346 ep 1 loss 0.5949

,run,method,view,attn,dev_F1,F1_argmax,P,R,thr,trainable_M,total_M,train_min,steps,gpu_GB,infer_tok_s
3,full,full,tail,causal-hybrid,0.7058,0.6787,0.7153,0.7082,-0.617,103.135,103.1,48.6,6035/9346,4.1,57918
5,partial,partial,tail,causal-hybrid,0.6951,0.6558,0.7066,0.6982,-0.460,31.317,103.1,41.0,9346/9346,1.6,60236
0,adalora,adalora,tail,causal-hybrid,0.6679,0.5905,0.6683,0.6680,-0.711,4.163,107.3,44.7,4707/14019,3.3,42618
1,adapter,adapter,tail,causal-hybrid,0.6648,0.6286,0.6685,0.6660,-0.519,2.394,105.5,50.1,7927/14019,2.5,56031
2,bitfit,bitfit,tail,causal-hybrid,0.6245,0.5322,0.6402,0.6308,-0.748,0.034,103.1,49.0,8082/14019,2.3,60873
4,lorma,lorma,tail,causal-hybrid,0.4645,0.3798,0.4812,0.4839,-0.063,3.234,106.4,46.1,6046/14019,3.1,45496


adalora.zip              root_files=['prediction.json'] n=1517 ones=54.1% ids_match_test=True md5=ee4d45f0
adapter.zip              root_files=['prediction.json'] n=1517 ones=43.0% ids_match_test=True md5=0d869f35
bitfit.zip               root_files=['prediction.json'] n=1517 ones=40.2% ids_match_test=True md5=b577ff98
full.zip                 root_files=['prediction.json'] n=1517 ones=41.9% ids_match_test=True md5=f8052bcc
lorma.zip                root_files=['prediction.json'] n=1517 ones=31.9% ids_match_test=True md5=1750fed5
partial.zip              root_files=['prediction.json'] n=1517 ones=41.0% ids_match_test=True md5=77b79c3d


### 11. Round 4 — knowledge distillation (teacher = `full`, 6-layer student) ‖ long-document representation (LoRMA over 4 chunks)

In [12]:
pA = launch(0, [("kd", X("kd", extra="--student_layers 6 --kd_T 2.0 --kd_alpha 0.5"), "full")])
pB = launch(1, [("lorma_chunks", f"--method lorma --epochs {EP['lorma']} --cw_pow 0.5 {CHUNKS} --rank {RANK} --alpha {2*RANK}", None)])
wait([pA, pB]); display(table()); collect()

[kd] running |   step 275/6848 ep 1 loss 0.5648 lr 9.98e-05 1.8min
[lorma_chunks] running |   step 50/1642 ep 1 loss 0.7679 lr 5.00e-04 1.4min
[kd] running |   step 575/6848 ep 1 loss 0.5052 lr 9.88e-05 3.8min
[lorma_chunks] running |   step 125/1642 ep 1 loss 0.8123 lr 9.99e-04 3.5min
[kd] running |   step 875/6848 ep 1 loss 0.4776 lr 9.68e-05 5.8min
[lorma_chunks] running |   step 200/1642 ep 1 loss 0.7980 lr 9.90e-04 5.7min
[kd] running |   step 1175/6848 ep 1 loss 0.4196 lr 9.39e-05 7.7min
[lorma_chunks] running |   step 275/1642 ep 1 loss 0.7526 lr 9.69e-04 7.8min
[kd] running |   step 1500/6848 ep 1 loss 0.4244 lr 8.98e-05 9.8min
[lorma_chunks] running |   step 350/1642 ep 1 loss 0.7452 lr 9.37e-04 9.9min
[kd] running |   step 1800/6848 ep 1 loss 0.4241 lr 8.52e-05 11.8min
[lorma_chunks] running |   step 400/1642 ep 1 loss 0.7427 lr 9.10e-04 11.3min
[kd] running |   step 2100/6848 ep 1 loss 0.3912 lr 7.99e-05 13.8min
[lorma_chunks] running |   step 475/1642 ep 1 loss 0.7276 lr 8.

,run,method,view,attn,dev_F1,F1_argmax,P,R,thr,trainable_M,total_M,train_min,steps,gpu_GB,infer_tok_s
3,full,full,tail,causal-hybrid,0.7058,0.6787,0.7153,0.7082,-0.617,103.135,103.1,48.6,6035/9346,4.1,57918
4,kd,kd,tail,causal-hybrid,0.6987,0.6668,0.7004,0.6992,-1.359,56.473,56.5,45.4,6848/14019,2.8,113637
7,partial,partial,tail,causal-hybrid,0.6951,0.6558,0.7066,0.6982,-0.460,31.317,103.1,41.0,9346/9346,1.6,60236
0,adalora,adalora,tail,causal-hybrid,0.6679,0.5905,0.6683,0.6680,-0.711,4.163,107.3,44.7,4707/14019,3.3,42618
1,adapter,adapter,tail,causal-hybrid,0.6648,0.6286,0.6685,0.6660,-0.519,2.394,105.5,50.1,7927/14019,2.5,56031
2,bitfit,bitfit,tail,causal-hybrid,0.6245,0.5322,0.6402,0.6308,-0.748,0.034,103.1,49.0,8082/14019,2.3,60873
6,lorma_chunks,lorma,chunks,causal-hybrid,0.5542,0.3333,0.5544,0.5543,-0.249,3.234,106.4,46.1,1642/14019,1.1,46721
5,lorma,lorma,tail,causal-hybrid,0.4645,0.3798,0.4812,0.4839,-0.063,3.234,106.4,46.1,6046/14019,3.1,45496


adalora.zip              root_files=['prediction.json'] n=1517 ones=54.1% ids_match_test=True md5=ee4d45f0
adapter.zip              root_files=['prediction.json'] n=1517 ones=43.0% ids_match_test=True md5=0d869f35
bitfit.zip               root_files=['prediction.json'] n=1517 ones=40.2% ids_match_test=True md5=b577ff98
full.zip                 root_files=['prediction.json'] n=1517 ones=41.9% ids_match_test=True md5=f8052bcc
kd.zip                   root_files=['prediction.json'] n=1517 ones=55.2% ids_match_test=True md5=d58ebb97
lorma.zip                root_files=['prediction.json'] n=1517 ones=31.9% ids_match_test=True md5=1750fed5
lorma_chunks.zip         root_files=['prediction.json'] n=1517 ones=52.6% ids_match_test=True md5=4a4d7ac1
partial.zip              root_files=['prediction.json'] n=1517 ones=41.0% ids_match_test=True md5=77b79c3d


### 12. Extra runs for the analysis (optional)
Examples: `--view head`, `--view headtail`, or `--pick spread` test which parts of the judgment matter; `--pool mean`, `--cw_pow 0`. These are analysis runs, not additional techniques.

In [13]:
pA = launch(0, [("lorma_head",     X("lorma", view="--view head --window 1024", extra=f"--rank {RANK} --alpha {2*RANK}"), None)])
pB = launch(1, [("lorma_headtail", X("lorma", view="--view headtail --window 1024 --head_tok 256", extra=f"--rank {RANK} --alpha {2*RANK}"), None)])
wait([pA, pB]); display(table()); collect()

[lorma_head] running |   step 250/6240 ep 1 loss 0.8557 lr 9.99e-04 1.8min
[lorma_headtail] running |   step 250/6089 ep 1 loss 0.8347 lr 9.98e-04 1.9min
[lorma_head] running |   step 500/6240 ep 1 loss 0.7792 lr 9.90e-04 3.7min
[lorma_headtail] running |   step 500/6089 ep 1 loss 0.7881 lr 9.89e-04 3.8min
[lorma_head] running |   step 775/6240 ep 1 loss nan lr 9.71e-04 5.8min
[lorma_headtail] running |   step 775/6089 ep 1 loss nan lr 9.69e-04 5.8min
[lorma_head] running |   step 1050/6240 ep 1 loss nan lr 9.42e-04 7.8min
[lorma_headtail] running |   step 1050/6089 ep 1 loss nan lr 9.39e-04 7.8min
[lorma_head] running |   step 1325/6240 ep 1 loss nan lr 9.05e-04 9.9min
[lorma_headtail] running |   step 1325/6089 ep 1 loss nan lr 9.00e-04 9.8min
[lorma_head] running |   step 1575/6240 ep 1 loss nan lr 8.64e-04 11.7min
[lorma_headtail] running |   step 1600/6089 ep 1 loss nan lr 8.53e-04 11.8min
[lorma_head] running |   step 1850/6240 ep 1 loss nan lr 8.13e-04 13.8min
[lorma_headtail] r

,run,method,view,attn,dev_F1,F1_argmax,P,R,thr,trainable_M,total_M,train_min,steps,gpu_GB,infer_tok_s
3,full,full,tail,causal-hybrid,0.7058,0.6787,0.7153,0.7082,-0.617,103.135,103.1,48.6,6035/9346,4.1,57918
4,kd,kd,tail,causal-hybrid,0.6987,0.6668,0.7004,0.6992,-1.359,56.473,56.5,45.4,6848/14019,2.8,113637
9,partial,partial,tail,causal-hybrid,0.6951,0.6558,0.7066,0.6982,-0.460,31.317,103.1,41.0,9346/9346,1.6,60236
0,adalora,adalora,tail,causal-hybrid,0.6679,0.5905,0.6683,0.6680,-0.711,4.163,107.3,44.7,4707/14019,3.3,42618
1,adapter,adapter,tail,causal-hybrid,0.6648,0.6286,0.6685,0.6660,-0.519,2.394,105.5,50.1,7927/14019,2.5,56031
2,bitfit,bitfit,tail,causal-hybrid,0.6245,0.5322,0.6402,0.6308,-0.748,0.034,103.1,49.0,8082/14019,2.3,60873
6,lorma_chunks,lorma,chunks,causal-hybrid,0.5542,0.3333,0.5544,0.5543,-0.249,3.234,106.4,46.1,1642/14019,1.1,46721
7,lorma_head,lorma,head,causal-hybrid,0.4967,0.3333,0.5088,0.5080,-0.589,3.234,106.4,47.8,6240/14019,3.1,45475
5,lorma,lorma,tail,causal-hybrid,0.4645,0.3798,0.4812,0.4839,-0.063,3.234,106.4,46.1,6046/14019,3.1,45496
8,lorma_headtail,lorma,headtail,causal-hybrid,0.3333,0.3333,0.2500,0.5000,0.000,3.234,106.4,45.9,6089/14019,3.1,46694


adalora.zip              root_files=['prediction.json'] n=1517 ones=54.1% ids_match_test=True md5=ee4d45f0
adapter.zip              root_files=['prediction.json'] n=1517 ones=43.0% ids_match_test=True md5=0d869f35
bitfit.zip               root_files=['prediction.json'] n=1517 ones=40.2% ids_match_test=True md5=b577ff98
full.zip                 root_files=['prediction.json'] n=1517 ones=41.9% ids_match_test=True md5=f8052bcc
kd.zip                   root_files=['prediction.json'] n=1517 ones=55.2% ids_match_test=True md5=d58ebb97
lorma.zip                root_files=['prediction.json'] n=1517 ones=31.9% ids_match_test=True md5=1750fed5
lorma_chunks.zip         root_files=['prediction.json'] n=1517 ones=52.6% ids_match_test=True md5=4a4d7ac1
lorma_head.zip           root_files=['prediction.json'] n=1517 ones=35.4% ids_match_test=True md5=ee5a1e66
lorma_headtail.zip       root_files=['prediction.json'] n=1517 ones=0.0% ids_match_test=True md5=1f8fe614
partial.zip              root_files=['

### 13. Ensembles (extra submissions) and the final checklist

In [14]:
os.makedirs(f"{WORK}/submissions", exist_ok=True)
def ens(out, extra):
    r = subprocess.run(f"python ensemble4_hybrid.py --runs_dir {WORK}/runs --out {WORK}/submissions/{out} {extra}", shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:] + r.stderr[-800:])
    if os.path.exists(f"{WORK}/submissions/{out}/submission.zip"): shutil.copy(f"{WORK}/submissions/{out}/submission.zip", f"{WORK}/submissions/{out}.zip")
ens("ens_all", "--all")
ens("ens_greedy", "--all --greedy")
for d in glob.glob(f"{WORK}/runs/*/submission.zip"):
    n = d.split("/")[-2]
    if not n.startswith("smoke") and n != "prep": shutil.copy(d, f"{WORK}/submissions/{n}.zip")
import zipfile, hashlib
test_ids = [str(r["id"]) if isinstance(r, dict) else None for r in json.load(open(f"{DATA}/test.json"))] if open(f"{DATA}/test.json").read(1) == "[" else None
for z in sorted(glob.glob(f"{WORK}/submissions/*.zip")):
    zf = zipfile.ZipFile(z); p = json.load(zf.open("prediction.json"))
    ok_ids = (test_ids is None) or ([str(x["id"]) for x in p] == test_ids)
    print(f"{os.path.basename(z):22s} files={zf.namelist()} n={len(p)} ones={100*np.mean([x['pred'] for x in p]):.1f}% ids_match_test={ok_ids} md5={hashlib.md5(zf.read('prediction.json')).hexdigest()[:8]}")

ensemble of ['adalora', 'adapter', 'bitfit', 'full', 'kd', 'lorma', 'lorma_chunks', 'lorma_head', 'lorma_headtail', 'partial']: dev macro-F1 0.3333 at threshold +0.000
wrote /kaggle/working/submissions/ens_all/submission.zip (1517 cases, 0.0% predicted 1)

  + full         -> dev macro-F1 0.7058
  + partial      -> dev macro-F1 0.7099
  + kd           -> dev macro-F1 0.7153
  + adalora      -> dev macro-F1 0.7176
  + lorma_head   -> dev macro-F1 0.7193
ensemble of ['full', 'partial', 'kd', 'adalora', 'lorma_head']: dev macro-F1 0.7193 at threshold -0.673
wrote /kaggle/working/submissions/ens_greedy/submission.zip (1517 cases, 46.0% predicted 1)

adalora.zip            files=['prediction.json'] n=1517 ones=54.1% ids_match_test=True md5=ee4d45f0
adapter.zip            files=['prediction.json'] n=1517 ones=43.0% ids_match_test=True md5=0d869f35
bitfit.zip             files=['prediction.json'] n=1517 ones=40.2% ids_match_test=True md5=b577ff98
ens_all.zip            files=['prediction.json

Identical `md5` values mean identical predictions and count only once toward the six submissions. Download `/kaggle/working/submissions/` and submit early; Codabench may take 15–30 minutes.